# Taller Interfaces Multimodales: Uniendo Voz y Gestos (Versión Tasks API)

Este cuaderno implementa una interfaz interactiva que combina la detección de gestos manuales (**MediaPipe Tasks API**) y el reconocimiento de voz (SpeechRecognition) para controlar una escena visual en tiempo real utilizando Pygame.

In [8]:
# Instalación de versiones específicas solicitadas
#!pip install numpy==1.26.4 mediapipe==0.10.11 opencv-python==4.9.0.80 speechrecognition==3.10.1 pyaudio==0.2.14 python-osc==1.8.3 pygame==2.5.2

# Descargar el modelo de detección de manos si no existe
import os
import urllib.request

model_path = 'hand_landmarker.task'

if not os.path.exists(model_path):
    print("Descargando modelo...")
    url = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
    urllib.request.urlretrieve(url, model_path)
    print("Modelo descargado.")

In [9]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import speech_recognition as sr
import pygame
import numpy as np
import threading
import queue
import time
from pythonosc import udp_client

# Configuración inicial ventana
WIDTH, HEIGHT = 800, 600
OSC_IP = "127.0.0.1"
OSC_PORT = 8000

# Inicializar cliente OSC
osc_client = udp_client.SimpleUDPClient(OSC_IP, OSC_PORT)

print("Librerías cargadas correctamente utilizando MediaPipe Tasks API.")

Librerías cargadas correctamente utilizando MediaPipe Tasks API.


## 1. Módulo de Reconocimiento de Voz

Hilo separado para comandos de voz.

In [10]:
voice_command_queue = queue.Queue()
last_command = "Ninguno"
listening_active = True

def speech_recognition_worker():
    global last_command, listening_active
    recognizer = sr.Recognizer()
    microphone = sr.Microphone()
    
    # Ajuste inicial al ruido ambiental
    with microphone as source:
        recognizer.adjust_for_ambient_noise(source, duration=1)
    
    print("Escuchando comandos...")
    
    while listening_active:
        try:
            with microphone as source:
                # Escuchar por un periodo corto
                audio = recognizer.listen(source, timeout=2, phrase_time_limit=3)
            
            # Intentar reconocer (Google API gratuita)
            command = recognizer.recognize_google(audio, language="es-ES").lower()
            print(f"Voz detectada: {command}")
            voice_command_queue.put(command)
            last_command = command
            
        except sr.WaitTimeoutError:
            continue
        except sr.UnknownValueError:
            continue
        except Exception as e:
            if listening_active:
                print(f"Error en voz: {e}")
            time.sleep(1)

# Iniciar el hilo de voz
voice_thread = threading.Thread(target=speech_recognition_worker, daemon=True)
voice_thread.start()

## 2. Módulo de Detección de Gestos (MediaPipe Tasks API)

Implementación del procesador de gestos sin usar `mediapipe.solutions`.

In [11]:
class GestureProcessor:
    def __init__(self, model_path='hand_landmarker.task'):
        # Configuración del Hand Landmarker
        base_options = python.BaseOptions(model_asset_path=model_path)
        options = vision.HandLandmarkerOptions(
            base_options=base_options,
            running_mode=vision.RunningMode.VIDEO,
            num_hands=1,
            min_hand_detection_confidence=0.5,
            min_hand_presence_confidence=0.5,
            min_tracking_confidence=0.5
        )
        self.landmarker = vision.HandLandmarker.create_from_options(options)
        
        # Conexiones de la mano para dibujo manual
        self.HAND_CONNECTIONS = [
            (0, 1), (1, 2), (2, 3), (3, 4), # Pulgar
            (0, 5), (5, 6), (6, 7), (7, 8), # Índice
            (5, 9), (9, 10), (10, 11), (11, 12), # Medio
            (9, 13), (13, 14), (14, 15), (15, 16), # Anular
            (13, 17), (0, 17), (17, 18), (18, 19), (19, 20) # Meñique
        ]

    def process_frame(self, frame, timestamp_ms):
        # Convertir BGR a RGB y crear imagen MP
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
        
        # Detectar
        result = self.landmarker.detect_for_video(mp_image, timestamp_ms)
        
        gesture = "Desconocido"
        landmarks = None
        
        if result.hand_landmarks:
            landmarks = result.hand_landmarks[0]
            self.draw_landmarks(frame, landmarks)
            gesture = self.classify_gesture(landmarks)
            
        return frame, gesture, landmarks

    def draw_landmarks(self, frame, landmarks):
        h, w, _ = frame.shape
        # Dibujar conexiones
        for connection in self.HAND_CONNECTIONS:
            start = landmarks[connection[0]]
            end = landmarks[connection[1]]
            cv2.line(frame, 
                     (int(start.x * w), int(start.y * h)), 
                     (int(end.x * w), int(end.y * h)), 
                     (0, 255, 0), 2)
        
        # Dibujar puntos
        for lm in landmarks:
            cv2.circle(frame, (int(lm.x * w), int(lm.y * h)), 5, (255, 0, 0), -1)

    def classify_gesture(self, landmarks):
        tips = [8, 12, 16, 20]
        pips = [6, 10, 14, 18]
        
        extended = []
        for t, p in zip(tips, pips):
            extended.append(landmarks[t].y < landmarks[p].y)
        
        # Pulgar (suponiendo mano derecha)
        thumb_extended = landmarks[4].x < landmarks[3].x
        
        count = extended.count(True)
        
        if count == 4 and thumb_extended:
            return "Mano Abierta"
        elif count == 0 and not thumb_extended:
            return "Puño"
        elif count == 2 and extended[0] and extended[1]:
            return "Dos Dedos"
        elif count == 1 and extended[0]:
            return "Indice"
        
        return "Otro"

## 3. Escena Reactiva Multimodal

Integración con Pygame utilizando el nuevo procesador.

In [12]:
def run_multimodal_app():
    global listening_active
    
    pygame.init()
    screen = pygame.display.set_mode((WIDTH, HEIGHT))
    pygame.display.set_caption("Multimodal Interface: Voice & Gestures (Tasks API)")
    clock = pygame.time.Clock()
    font = pygame.font.SysFont("Arial", 24)
    
    obj_color = [200, 200, 200]
    obj_pos = [WIDTH // 2, HEIGHT // 2]
    obj_rotation = 0
    obj_visible = True
    
    cap = cv2.VideoCapture(0)
    processor = GestureProcessor()
    
    start_time = time.time()
    running = True
    while running:
        screen.fill((20, 20, 30))
        
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
        
        ret, frame = cap.read()
        if not ret: break
        
        frame = cv2.flip(frame, 1)
        timestamp_ms = int((time.time() - start_time) * 1000)
        frame, current_gesture, landmarks = processor.process_frame(frame, timestamp_ms)
        
        current_voice = ""
        while not voice_command_queue.empty():
            current_voice = voice_command_queue.get()
        
        status_msg = "Esperando entrada..."
        
        if current_gesture == "Mano Abierta":
            if "azul" in current_voice:
                obj_color = [50, 50, 255]
                status_msg = "Acción: Color cambiado a AZUL"
            if "amarillo" in current_voice:
                obj_color = [255, 255, 50]
                status_msg = "Acción: Color cambiado a AMARILLO"
            elif "verde" in current_voice:
                obj_color = [50, 255, 50]
                status_msg = "Acción: Color cambiado a VERDE"
            elif "rojo" in current_voice:
                obj_color = [255, 50, 50]
                status_msg = "Acción: Color cambiado a ROJO"
        
        if current_gesture in ["Dos Dedos", "Indice"] and "mover" in last_command:
            if landmarks:
                obj_pos[0] = int(landmarks[8].x * WIDTH)
                obj_pos[1] = int(landmarks[8].y * HEIGHT)
                status_msg = "Acción: Moviendo objeto"
        
        if "rotar" in current_voice:
            obj_rotation += 45
            status_msg = "Acción: Rotando"
            
        if "ocultar" in current_voice:
            obj_visible = False
        elif "mostrar" in current_voice:
            obj_visible = True
            
        if obj_visible:
            rect_surface = pygame.Surface((100, 100), pygame.SRCALPHA)
            pygame.draw.rect(rect_surface, obj_color, (0, 0, 100, 100))
            rotated_rect = pygame.transform.rotate(rect_surface, obj_rotation)
            rect_dest = rotated_rect.get_rect(center=obj_pos)
            screen.blit(rotated_rect, rect_dest)
        
        # HUD
        cv2.putText(frame, f"Gesto: {current_gesture}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
        cv2.putText(frame, f"Voz: {last_command}", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 0), 2)
        
        frame_small = cv2.resize(frame, (240, 180))
        frame_pygame = pygame.image.frombuffer(frame_small.tobytes(), frame_small.shape[1::-1], "BGR")
        screen.blit(frame_pygame, (WIDTH - 250, 10))
        
        gesture_txt = font.render(f"Gesto: {current_gesture}", True, (255, 255, 255))
        voice_txt = font.render(f"Voz: {last_command}", True, (255, 255, 255))
        status_txt = font.render(status_msg, True, (0, 255, 255))
        
        screen.blit(gesture_txt, (20, 20))
        screen.blit(voice_txt, (20, 50))
        screen.blit(status_txt, (20, HEIGHT - 40))
        
        pygame.display.flip()
        clock.tick(30)
        
        osc_client.send_message("/gesture", current_gesture)
        if current_voice:
            osc_client.send_message("/voice", current_voice)

    cap.release()
    pygame.quit()
    listening_active = False

print("Función de aplicación actualizada con Tasks API.")

Función de aplicación actualizada con Tasks API.


In [13]:
try:
    run_multimodal_app()
except Exception as e:
    print(f"Error al ejecutar: {e}")
finally:
    listening_active = False

Escuchando comandos...
Voz detectada: rojo
Voz detectada: mover
Voz detectada: azul
Voz detectada: verde
Voz detectada: mover
